# Lab - Validate Business Rules, Retry with the Error

Three tasks on not trusting the values Claude returns. A well-formed answer with a wrong value does more damage than no answer: it gets stored. The schema checks the shape; your code checks the values against the business.

| Task | What you learn |
| --- | --- |
| 1 | Allow null, so a missing value travels honestly into storage |
| 2 | Validation rules that return exact errors - format and existence |
| 3 | A bounded retry that sends the error back, and records what happened |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic
import json
import re

load_dotenv()
client = Anthropic()

# The model name lives in one constant, so a later update changes one line.
MODEL = "claude-sonnet-5-5"

INSTRUCTION = (
    "Extract the return request from this customer email. "
    "Order IDs look like ORD-12345678. If the email gives no "
    "order number, set order_id to null. Never invent one."
)

ORDERS = {"ORD-12345678", "ORD-87654321", "ORD-11112222"}   # the orders that exist

EMAILS = [
    "Hi, I want to return the blue jacket from order "
    "ORD-12345678. It arrived last week but it is too small. Alex",
    "Hello, the shoes I got are damaged. Please tell me how "
    "to send them back. Thanks, Sam",
    "The headphones from order ORD-7788 stopped working after "
    "two days. I want a return. Kim",
    "My scarf from order ORD-99999999 is damaged. I want to send it back. Lee",
]

---

## Task 1 - Null is an honest answer

In [ ]:
# ============================================================
# TASK 1 - Null is an honest answer
# ============================================================
#
# WHAT TO DO
#   Let order_id be a string or null, and send the instruction
#   with every email. Extract the first two emails, and save
#   the records as JSON.
#
# WHY IT MATTERS
#   Last time the email without an order number came back with
#   order_id "unknown" - a string, so the schema accepted it.
#   The first fix is in the schema: allow null, and tell Claude
#   what an order ID looks like and never to invent one. A null
#   travels through your whole system honestly: None in Python,
#   null in the stored JSON, an empty cell in a report.
#
# WHERE TO SEE IT IN THE LECTURE
#   "The first fix is in the schema itself".
#
# HOW TO DO IT
#   Three blanks:
#
#       "null"        the second type order_id may have
#       INSTRUCTION   the rules for Claude, before the email
#       records       the list to save as JSON
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    SCHEMA = {
        "type": "object",
        "properties": {
            "order_id": {"type": ["string", ...]},   # "null"
            "item": {"type": "string"},
            "reason": {"type": "string",
                       "enum": ["too_small", "too_big", "damaged", "other"]},
        },
        "required": ["order_id", "item", "reason"],
        "additionalProperties": False,
    }
    
    def request(messages):
        response = client.messages.create(
            model=MODEL, max_tokens=1000,
            output_config={"effort": "low",
                           "format": {"type": "json_schema", "schema": SCHEMA}},
            messages=messages)
        return next(b.text for b in response.content if b.type == "text")
    
    records = [json.loads(request([{"role": "user", "content": ... + "\n\n" + email}]))   # INSTRUCTION
               for email in EMAILS[:2]]
    saved = json.dumps(...)   # records
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
print(saved)

check("honest_null", schema=SCHEMA, records=records, saved=saved)

---

## Task 2 - Rules that return the exact error

In [ ]:
# ============================================================
# TASK 2 - Rules that return the exact error
# ============================================================
#
# WHAT TO DO
#   Write the validation as a list of rules. Each rule looks at
#   one thing and returns an exact error message - or None when
#   the data is fine. validate() returns the first error it
#   finds.
#
# WHY IT MATTERS
#   Values can be wrong in ways the schema cannot see. A typo or
#   a four-digit number breaks the format. Worse, a number in
#   the right format may not be an order at all - only your
#   data knows that. And the message must be exact - which
#   field, which value, what is wrong - because the retry sends
#   it to Claude.
#
# WHERE TO SEE IT IN THE LECTURE
#   "So we add a validation function in code".
#
# HOW TO DO IT
#   Three blanks:
#
#       r"ORD-\d{8}"   the format of a real order ID
#       ORDERS          the orders that exist
#       RULES           every rule, in order
# ============================================================

# TODO: replace each ... below with the value named beside it
def order_id_format(data):
    order_id = data["order_id"]
    if order_id is not None and not re.fullmatch(..., order_id):   # r"ORD-\d{8}"
        return f"order_id {order_id!r} is not a valid order ID"


def order_exists(data):
    order_id = data["order_id"]
    if order_id is not None and order_id not in ...:   # ORDERS
        return f"order_id {order_id!r} is not an order in our system"


RULES = [order_id_format, order_exists]


def validate(data):
    for rule in ...:   # RULES
        error = rule(data)
        if error:
            return error
    return None

try:
    for order_id in ("ORD-12345678", None, "ORD-7788", "ORD-99999999"):
        print(repr(order_id), "->", validate({"order_id": order_id}) or "fine")
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc) or "must be string" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise

check("business_rules", validate=validate)

---

## Task 3 - A bounded retry, with a record of it

In [ ]:
# ============================================================
# TASK 3 - A bounded retry, with a record of it
# ============================================================
#
# WHAT TO DO
#   Turn extraction into a loop of up to three attempts. Each
#   attempt sends the conversation, parses, validates. Fine:
#   return the data. Not fine: add Claude's answer and the exact
#   error, and try again. Every result records its attempts and
#   errors.
#
# WHY IT MATTERS
#   Retrying the same request and hoping is a coin flip. Sending
#   the exact error back turns the retry into a correction - and
#   bounding it means a hopeless case cannot loop forever. The
#   record of attempts and errors is what you watch in
#   production: more retries means the prompt or the data got
#   worse.
#
# WHERE TO SEE IT IN THE LECTURE
#   "So the practical rule: let the schema shape the answer".
#
# HOW TO DO IT
#   Three blanks:
#
#       3             the most attempts - always bounded
#       "assistant"   Claude's own answer goes back as its turn
#       error         the exact message validate() returned
# ============================================================

try:
    # TODO: replace each ... below with the value named beside it
    def extract(email):
        messages = [{"role": "user", "content": INSTRUCTION + "\n\n" + email}]
        errors = []
        for attempt in range(1, ... + 1):   # 3 - the most attempts
            text = request(messages)
            data = json.loads(text)
            error = validate(data)
            if error is None:
                return {"data": data, "attempts": attempt, "errors": errors}
            errors.append(error)
            messages.append({"role": ..., "content": text})   # "assistant"
            messages.append({"role": "user", "content": ... + ". Fix it, or use null."})   # error
        return {"data": None, "attempts": attempt, "errors": errors}
    
    results = [extract(email) for email in EMAILS]
except (TypeError,) as exc:   # provided: a blank left above, said plainly
    if "llipsis" in str(exc):
        raise ValueError("You still have ... above. Replace each one with the value "
                         "named beside it.") from None
    raise
for result in results:
    print(result["attempts"], "attempt(s)", result["data"], result["errors"] or "")

check("retry_record", results=results)

---

## Done

Let the schema shape the answer, let your code validate the values - against the format and against your own data - and when validation fails, send the exact error back and retry a bounded number of times. Keep the record: attempts and errors are what you monitor.